# 04 - Modelo de risco de review baixo

In [ ]:
from pathlib import Path
import sys
import warnings

import joblib
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from olist.config import FIGURES_DIR, MODELS_DIR, PROCESSED_DATA_DIR, REPORTS_DIR, ensure_project_dirs
from olist.plots import save_current_figure, set_plot_style

ensure_project_dirs()
set_plot_style()

In [ ]:
parquet_path = PROCESSED_DATA_DIR / 'modeling_dataset.parquet'
csv_path = PROCESSED_DATA_DIR / 'modeling_dataset.csv'

if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
else:
    df = pd.read_csv(csv_path)

df = df.dropna(subset=['is_low_review']).copy()
df['is_low_review'] = df['is_low_review'].astype(int)

print(df.shape)
print('Taxa de review baixo:', df['is_low_review'].mean().round(4))
display(df.head())

## Separação treino/teste

Usei estratificação para preservar a proporção de reviews baixos. Em uma versão mais avançada, uma validação temporal seria mais realista.

In [ ]:
target = 'is_low_review'
ignore_cols = ['order_id', 'customer_id', 'customer_unique_id', 'review_score', target]

feature_cols = [col for col in df.columns if col not in ignore_cols]
categorical_features = [col for col in feature_cols if df[col].dtype == 'object' or str(df[col].dtype).startswith('string')]
numeric_features = [col for col in feature_cols if col not in categorical_features]

X = df[feature_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Numericas:', numeric_features)
print('Categoricas:', categorical_features)

In [ ]:
def make_preprocessor():
    numeric_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])

    categorical_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=50)),
    ])

    return ColumnTransformer(transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features),
    ])

models = {
    'logistic_regression': Pipeline(steps=[
        ('preprocess', make_preprocessor()),
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
    ]),
    'random_forest': Pipeline(steps=[
        ('preprocess', make_preprocessor()),
        ('model', RandomForestClassifier(n_estimators=250, min_samples_leaf=20, class_weight='balanced_subsample', random_state=42, n_jobs=-1)),
    ]),
}

metrics = []
fitted_models = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    proba = pipeline.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    metrics.append({
        'model': name,
        'roc_auc': roc_auc_score(y_test, proba),
        'average_precision': average_precision_score(y_test, proba),
        'positive_rate_at_050': pred.mean(),
    })
    fitted_models[name] = pipeline

metrics_df = pd.DataFrame(metrics).sort_values('average_precision', ascending=False)
display(metrics_df)
metrics_df.to_csv(REPORTS_DIR / 'model_metrics.csv', index=False)

###### Mesmo sem acesso as informações finais da entrega, o modelo conseguiu identificar sinais antecipados de risco associados à pior experiência do cliente.

## Escolha de threshold

In [ ]:
best_model_name = metrics_df.iloc[0]['model']
best_model = fitted_models[best_model_name]
proba_test = best_model.predict_proba(X_test)[:, 1]

threshold_rows = []
for threshold in np.arange(0.10, 0.91, 0.05):
    pred = (proba_test >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, pred, average='binary', zero_division=0)
    threshold_rows.append({
        'threshold': round(float(threshold), 2),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'flagged_rate': pred.mean(),
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)
threshold_df.to_csv(REPORTS_DIR / 'threshold_analysis.csv', index=False)

sns.set_theme(style='whitegrid')


fig, ax = plt.subplots(figsize=(10, 6))

sns.lineplot(
    data=threshold_df,
    x='threshold',
    y='recall',
    marker='o',
    linewidth=2.5,
    markersize=8,
    color='#4C78A8',
    label='Recall',
    ax=ax
)

sns.lineplot(
    data=threshold_df,
    x='threshold',
    y='precision',
    marker='o',
    linewidth=2.5,
    markersize=8,
    color='#E45756',
    label='Precision',
    ax=ax
)

sns.lineplot(
    data=threshold_df,
    x='threshold',
    y='flagged_rate',
    marker='o',
    linewidth=2.5,
    markersize=8,
    color='#72B7B2',
    label='Flagged rate',
    ax=ax
)


ax.set_title(
    f'AnÃ¡lise de Threshold â€” {best_model_name}',
    fontsize=16,
    weight='bold',
    pad=18
)


ax.set_xlabel(
    'Threshold de classificaÃ§Ã£o',
    fontsize=11
)

ax.set_ylabel(
    'Percentual',
    fontsize=11
)

ax.yaxis.set_major_formatter(PercentFormatter(1))


ax.grid(axis='y', alpha=0.25)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

legend = ax.legend(
    title='MÃ©trica',
    frameon=False
)

legend.get_title().set_fontsize(11)

best_row = threshold_df.loc[
    threshold_df['precision'].idxmax()
]

ax.axvline(
    best_row['threshold'],
    linestyle='--',
    color='gray',
    alpha=0.7
)

ax.text(
    best_row['threshold'] + 0.01,
    0.05,
    f"Melhor precision\nThreshold={best_row['threshold']:.2f}",
    fontsize=9,
    color='gray'
)

plt.tight_layout()

save_current_figure(
    FIGURES_DIR / '01_h5_threshold_analysis.png'
)

plt.show()

######
- Com um threshold de 0.3 podemos observar que:
    1. O modelo encontra quase todos os casos ruins.
    2. Mas praticamente marca todos os pedidos como risco.
    3. EntÃ£o operacionalmente isso seria inviável.

- Com um threshold de 0.5 podemos observar que:
    1. Tenho um resultado muito mais razoável.
    2. O sistema sinaliza:
        1. (~21%) dos pedidos.
        2. encontra (~41%) do reviews ruins.
        3. com precisão 2 x maior que a taxa base (que é ~12%)
    3. Então escolher pedidos aleatários teria precisão de 12%.
    4. O modelo consegue precisão de (~25%). Ou seja: Ele dobra a eficiência operacional

Conclusão:
O threshold ideal depende do custo operacional do erro. Um threshold menor aumenta a capacidade de detectar pedidos problemáticos, mas também eleva a quantidade de falsos positivos. Já thresholds maiores tornam as previsões mais precisas, porém deixam escapar parte dos casos críticos.

In [ ]:
chosen_threshold = 0.50
pred_test = (proba_test >= chosen_threshold).astype(int)

print(classification_report(y_test, pred_test, digits=3))
cm = confusion_matrix(y_test, pred_test)

sns.set_theme(style='white')

fig, ax = plt.subplots(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    cbar=False,
    linewidths=1,
    linecolor='white',
    square=True,
    annot_kws={
        'fontsize': 14,
        'weight': 'bold'
    },
    ax=ax
)

ax.set_title(
    f'Matriz de ConfusÃ£o â€” Threshold {chosen_threshold:.2f}',
    fontsize=16,
    weight='bold',
    pad=18
)

ax.set_xlabel(
    'Classe predita',
    fontsize=11
)

ax.set_ylabel(
    'Classe real',
    fontsize=11
)

ax.set_xticklabels(
    ['Review positivo', 'Review negativo'],
    fontsize=10
)

ax.set_yticklabels(
    ['Review positivo', 'Review negativo'],
    fontsize=10,
    rotation=0
)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()

save_current_figure(
    FIGURES_DIR / '02_h5_confusion_matrix.png'
)

plt.show()

######

- Interpretação das métricas:
    1. Precision = 25%
       1. De cada 4 pedidos marcados como risco, 1 realmente recebe review ruim.
    2. Recall = 41.5%
       1. O modelo consegue capturar cerca de 4 em cada 10 pedidos problemáticos.
          
- Sem o modelo seria necessário acompanhar TODOS os pedidos.
- Com o modelo:
    1. É necessário acompanhar (~21%).
    2. Dos quais (~41%) dos casos ruins são detectados.

Conclusão:
O modelo não foi construí­do para substituir completamente a análise operacional, mas para priorizar pedidos com maior probabilidade de insatisfação, reduzindo o volume de monitoramento necessário e aumentando a eficiência da detecão de casos crí­ticos.

In [ ]:
model_artifact = {
    'model_name': best_model_name,
    'pipeline': best_model,
    'feature_cols': feature_cols,
    'categorical_features': categorical_features,
    'numeric_features': numeric_features,
    'chosen_threshold': chosen_threshold,
}

joblib.dump(model_artifact, MODELS_DIR / 'low_review_risk_model.joblib')
print(f'Modelo salvo em: {MODELS_DIR / "low_review_risk_model.joblib"}')